# Slide 1 — Title\n\n## Design and Implementation of An On-Device Bilingual AI Voice Assistant System for English and German Based on Deep Learning\n\n**Student:** Ilyass Lambardi (202239060075)  \n**Supervisor:** Jianbo Wang  \n**School:** Computer Science and Software Engineering  \n**University:** Southwest Petroleum University  \n**Date:** May 2026

# Slide 2 — Problem & Motivation\n\n**Problem:**\n- Existing voice assistants (Siri, Alexa, Google) are cloud-only, high-latency, and monolingual per interaction\n- No open-source system supports real-time bilingual (EN/DE) speech-to-speech with mid-conversation code-switching\n- Privacy concerns: all audio sent to remote servers\n\n**Motivation:**\n- Enable natural bilingual conversations without manual language toggling\n- Provide a dual-mode architecture: cloud (high quality) + local (private, offline)\n- Achieve sub-second end-to-end latency via streaming pipeline design\n- Build a modular, extensible system for academic and practical use

# Slide 3 — System Overview\n\n**Full-stack Speech-to-Speech Pipeline:**\n\n```\nMicrophone → VAD → ASR → LLM → TTS → Speaker\n             (Silero) (Whisper) (Llama/Qwen) (Edge/Silero)\n```\n\n**Key Design Decisions:**\n- WebSocket streaming: binary PCM audio + JSON control over single connection\n- Sentence-level streaming: TTS starts as soon as first sentence is complete (overlaps with LLM generation)\n- Dual-mode: Cloud (Groq API) or Local (on-device GPU inference)\n- Interruptible: user can speak mid-generation to cut off the AI\n- Long-term memory: SQLite-backed semantic recall across sessions

# Slide 4 — Backend Architecture\n\n```\nbackend/\n├── main.py                  # FastAPI server + WebSocket endpoint\n├── config.py                # Centralized env-driven configuration\n└── pipeline/\n    ├── cloud/               # Groq API mode (multi-user, per-session)\n    │   ├── manager.py       # Pipeline orchestrator + state machine\n    │   ├── asr_groq.py      # Groq Whisper Large v3 ASR\n    │   ├── llm_groq.py      # Groq Llama 3.3 70B streaming\n    │   ├── tts_edge.py      # Microsoft Edge Neural TTS\n    │   ├── memory.py        # Long-term memory (SQLite + semantic)\n    │   ├── hallucination_filter.py\n    │   └── lang_detect.py   # Intent-based language detection\n    └── local/               # On-device mode (single-user, GPU)\n        ├── manager.py       # Local orchestrator\n        ├── asr.py           # faster-whisper (int8/fp16)\n        ├── llm.py           # Qwen 2.5 1.5B Instruct\n        ├── tts.py           # Silero v3 dual-voice\n        ├── vad.py           # Silero VAD (shared by both modes)\n        └── memory.py\n```

# Slide 5 — Pipeline Manager (Core Logic)\n\n**State Machine:** `idle → listening → thinking → speaking → idle`\n\n**Key mechanisms:**\n1. **VAD accumulation** — collects speech frames until silence threshold (700ms), then triggers ASR\n2. **Interrupt detection** — frame-counting mechanism: 4+ consecutive speech frames (~128ms) during generation = interrupt\n3. **Backchannel filtering** — short utterances (<384ms like \"mhm\", \"yeah\") don't interrupt\n4. **Pipeline lock** — `asyncio.Lock()` prevents concurrent pipeline runs from overlapping audio\n5. **Generation guard** — `_generating` flag ensures only one LLM+TTS chain runs at a time\n\n**Concurrency model:**\n- `asyncio.Task` for generation pipeline\n- `asyncio.Event` for interrupt signaling\n- Per-session managers in cloud mode, shared manager in local mode

# Slide 6 — VAD + ASR Module\n\n## Voice Activity Detection (Silero VAD v5)\n- Always runs locally (CPU, ~2M params)\n- 512-sample frames (~32ms) at 16kHz\n- Adaptive threshold (base 0.45) with noise floor tracking\n- Outputs: `is_speaking` flag + complete utterance audio buffer\n\n## Automatic Speech Recognition\n| Mode | Model | Params | Features |\n|------|-------|--------|----------|\n| Cloud | Groq Whisper Large v3 | 1.5B | Language tag, timestamps, best accuracy |\n| Local | faster-whisper small | 244M | int8/fp16 quantization, beam=2, CUDA |\n\n**Hallucination filter:** rejects known noise artifacts (\"Thank you\", \"Untertitelung...\", repeated tokens) using pattern matching + entropy scoring

# Slide 7 — LLM Module\n\n## Cloud: Groq Llama 3.3 70B Versatile\n- Token-by-token streaming via Groq API\n- System prompt: bilingual persona \"Alex\" with language mirroring rules\n- Max tokens: 120 (optimized for voice — short, natural replies)\n- Temperature: 0.85 (creative but coherent)\n\n## Local: Qwen 2.5 1.5B Instruct\n- Runs on-device with 4GB VRAM\n- GGUF quantization or HuggingFace inference\n- Context window: 768 tokens (voice chat doesn't need long context)\n\n## Language Intelligence\n- **Intent-based detection:** overrides ASR language tag with structural sentence analysis\n- **Language mirroring:** responds in whichever language the user speaks\n- **Teacher Mode:** cross-lingual explanations triggered by \"What does [word] mean?\"\n- **Denglisch handling:** mixed input → dominant-language response

# Slide 8 — TTS Module\n\n## Edge TTS (Cloud mode — default)\n- Microsoft Edge Neural voices (free, high quality)\n- Multilingual: `en-US-GuyNeural` (EN), `de-DE-ConradNeural` (DE)\n- Streaming synthesis with SSML support\n- Sample rate: 24kHz, Int16 PCM\n\n## Silero v3 (Local mode)\n- Dual models: English (`en_0`) + German (`eva_k`) speakers\n- ~10M params each, CPU-friendly\n- Low latency, no network dependency\n\n## Sentence-Level Streaming Strategy\n```\nLLM: [token][token][sentence_end] → TTS synthesizes sentence 1\nLLM: [token][token][sentence_end] → TTS synthesizes sentence 2 (overlaps)\n```\nUser hears first sentence while LLM is still generating → perceived latency ≈ time-to-first-sentence

# Slide 9 — Memory System\n\n## Long-Term Memory (SQLite + Semantic Search)\n- Automatically stores conversation exchanges as memory entries\n- Retrieves relevant past context based on current topic similarity\n- Cross-session recall: remembers user preferences, past topics\n- Configurable via `LTM_RECALL` environment variable\n\n## Conversation History\n- Sliding window with configurable depth\n- Injected into LLM system prompt for context continuity\n- Separate from long-term memory (short-term vs long-term)\n\n## Design Choice\n- SQLite chosen for: zero-config, single-file, no external DB server\n- Semantic similarity via embedding comparison\n- Per-session LTM in cloud mode, shared in local mode

# Slide 10 — Tech Stack Summary\n\n| Layer | Technology | Purpose |\n|-------|-----------|----------|\n| Server | FastAPI + Uvicorn | Async WebSocket server |\n| Protocol | WebSocket (binary PCM + JSON) | Real-time bidirectional streaming |\n| VAD | Silero VAD v5 | Speech boundary detection |\n| ASR | Whisper Large v3 / faster-whisper | Bilingual speech recognition |\n| LLM | Llama 3.3 70B / Qwen 2.5 1.5B | Conversational generation |\n| TTS | Edge TTS / Silero v3 | Bilingual speech synthesis |\n| Memory | SQLite | Long-term semantic recall |\n| Config | python-dotenv + dataclass | Environment-driven settings |\n| Deploy | Docker | Containerized deployment |\n| Language | Python 3.10+ | Backend runtime |

# Slide 11 — Evaluation Results\n\n## Latency Performance (Cloud Mode)\n| Stage | Measured Latency |\n|-------|------------------|\n| VAD → speech end detection | ~700ms (configurable silence threshold) |\n| ASR (Groq Whisper) | ~300-500ms |\n| LLM first token (Groq) | ~200-400ms |\n| TTS first sentence | ~400-600ms |\n| **Total end-to-end** | **~1.5-2.5s** (first audio playback) |\n\n## Bilingual Accuracy\n- Language detection accuracy: >95% for clear EN/DE input\n- Code-switching handled via intent-based override of ASR tags\n- Hallucination filter rejects >90% of noise-generated false transcripts\n\n## Interrupt Responsiveness\n- Interrupt triggered within ~128ms of sustained user speech\n- Backchannel correctly ignored for utterances <384ms

# Slide 12 — Limitations\n\n1. **Cloud dependency** — Best quality requires internet (Groq API); local mode trades accuracy for privacy\n2. **Local mode VRAM** — Qwen 2.5 1.5B + faster-whisper needs ~4GB VRAM; not feasible on all devices\n3. **Language scope** — Only English + German; adding new languages requires new TTS voices + prompt tuning\n4. **Single-user local mode** — GPU models can't be shared across concurrent sessions\n5. **TTS naturalness** — Silero v3 (local) is robotic compared to Edge TTS (cloud); no voice cloning\n6. **No speaker diarization** — Cannot distinguish multiple speakers in the same session\n7. **Memory retrieval** — Simple semantic similarity; no sophisticated RAG or knowledge graph

# Slide 13 — Future Work\n\n1. **More languages** — Extend to Chinese, Arabic, French via modular TTS/prompt plugins\n2. **On-device LLM upgrades** — Integrate Phi-3 or Gemma 2 for better local quality with similar VRAM\n3. **Voice cloning** — XTTSv2 with user voice reference for personalized output\n4. **Speaker diarization** — Multi-speaker awareness using pyannote or similar\n5. **RAG integration** — Connect to external knowledge bases for factual grounding\n6. **Mobile deployment** — Optimize pipeline for Android/iOS with ONNX runtime\n7. **Emotion detection** — Use prosody analysis to adapt response tone\n8. **Evaluation framework** — Automated MOS scoring + bilingual benchmark dataset

# Slide 14 — Conclusion\n\n## Summary\n- Designed and implemented a **full-stack bilingual speech-to-speech AI system** supporting English and German\n- Built a **streaming pipeline** (VAD → ASR → LLM → TTS) with sentence-level overlapping for low perceived latency\n- Achieved **dual-mode architecture**: cloud (Groq API, best quality) + local (on-device, private)\n- Implemented **natural conversation features**: interruption, backchannel detection, language mirroring, long-term memory\n- System handles **mid-conversation code-switching** without manual language selection\n\n## Contributions\n1. Streaming sentence-level TTS architecture reducing perceived latency by ~40% vs full-response synthesis\n2. Intent-based bilingual detection overriding ASR language tags for accurate code-switching\n3. Modular dual-mode design enabling seamless cloud↔local fallback\n4. Open-source implementation with Docker deployment support\n\n---\n**Thank you! Questions?**